In [1]:
import polars as pl
import pandas as pd
import numpy as np
import catboost
import os
from datetime import date, timedelta

In [2]:
test_start_date = date(2024, 8, 1)
val_start_date = date(2024, 7, 1)
val_end_date = date(2024, 7, 31)
train_end_date = date(2024, 6, 30)
data_path = './predict-user-fresh-order/'

# Read data

In [3]:
actions_history = pl.read_parquet(os.path.join(data_path, 'actions_history'))
search_history = pl.read_parquet(os.path.join(data_path, 'search_history'))
product_information = pl.read_csv(os.path.join(data_path, 'product_information.csv'))

In [4]:
product_information.head(100)

product_id,name,brand,type,category_id,category_name,price,discount_price
i64,str,str,str,i64,str,f64,f64
26176363,"""Развивающие тесты (3-4 года) (…","""Machaon""","""Печатная книга: Развитие детей""",780,"""Книги""",380.0,274.0
29898500,"""Mexx Туалетная вода Ice Touch …","""Mexx""","""Туалетная вода""",117,"""Мужская""",2645.0,1859.0
33967827,"""64 ГБ USB Флеш-накопитель USB …","""SmartBuy""","""USB-флеш-накопитель""",178,"""Флешки и CD-R""",1690.0,469.0
135938830,"""Чай листовой чёрный Ahmad Tea …","""Ahmad Tea""","""Чай листовой""",465,"""Чай листовой""",319.0,244.0
137920686,"""Seagate 4 ТБ Внешний жесткий д…","""Seagate""","""Внешний жесткий диск""",615,"""Внешние жесткие диски""",28590.0,9539.0
…,…,…,…,…,…,…,…
499019785,"""Набор для игры в покер ""Poker …","""Анзор Игра""","""Набор для покера""",5,"""Настольные игры для детей""",999.0,425.0
519410131,"""Органайзер для хранения / Орга…","""Baona""","""Органайзер для вещей""",37,"""Товары для дачи""",1490.0,838.0
524083710,"""Футляр для очков на магните му…","""O.Rissa""","""Футляр для очков""",56,"""Очки и аксессуары""",850.0,407.0


In [5]:
pl.read_csv(os.path.join(data_path, 'action_type_info.csv'))

action_type,action_type_id
str,i64
"""click""",1
"""favorite""",2
"""order""",3
"""search""",4
"""to_cart""",5
"""view""",6


In [6]:
val_start_date, val_end_date

(datetime.date(2024, 7, 1), datetime.date(2024, 7, 31))

In [7]:
val_target = (
    actions_history
    .filter(pl.col('timestamp').dt.date() >= val_start_date)
    .filter(pl.col('timestamp').dt.date() <= val_end_date)
    .select('user_id', (pl.col('action_type_id') == 3).alias('has_order'))
    .group_by('user_id')
    .agg(pl.max('has_order').cast(pl.Int32).alias('target'))
)

In [8]:
val_target.group_by('target').agg(pl.count('user_id'))

target,user_id
i32,u32
0,1227381
1,647575


In [9]:
val_target.group_by('target').agg(pl.count('user_id')).collect_schema()

Schema([('target', Int32), ('user_id', UInt32)])

# Simple pipeline

## Feats

In [10]:
train_end_date, train_end_date - timedelta(days=30 * 4)

(datetime.date(2024, 6, 30), datetime.date(2024, 3, 2))

In [11]:
actions_aggs = {}
actions_id_to_suf = {
    1: "click",
    2: "favorite",
    3: "order",
    5: "to_cart",
}
for id_, suf in actions_id_to_suf.items():
    aggs = (
        actions_history
        .filter(pl.col('timestamp').dt.date() <= train_end_date)
        .filter(pl.col('timestamp').dt.date() >= train_end_date - timedelta(days=30 * 4))
        .filter(pl.col('action_type_id') == id_)
        .join(
            product_information
            .select('product_id', 'discount_price'),
            on='product_id',
        )
        .group_by('user_id')
        .agg(
            pl.count('product_id').cast(pl.Int32).alias(f'num_products_{suf}'),
            pl.sum('discount_price').cast(pl.Float32).alias(f'sum_discount_price_{suf}'),
            pl.max('discount_price').cast(pl.Float32).alias(f'max_discount_price_{suf}'),
            pl.max('timestamp').alias(f'last_{suf}_time'),
            pl.min('timestamp').alias(f'first_{suf}_time'),
        )
        .with_columns([
            (pl.lit(val_start_date) - pl.col(f'last_{suf}_time'))
            .dt.total_days()
            .cast(pl.Int32)
            .alias(f'days_since_last_{suf}'),
            
            (pl.lit(val_start_date) - pl.col(f'first_{suf}_time'))
            .dt.total_days()
            .cast(pl.Int32)
            .alias(f'days_since_first_{suf}'),
        ])
        .select(
            'user_id',
            f'num_products_{suf}',
            f'sum_discount_price_{suf}',
            f'max_discount_price_{suf}',
            f'days_since_last_{suf}',
            f'days_since_first_{suf}',
        )
    )
    actions_aggs[id_] = aggs

In [12]:
# search_aggs
id_ = 4
suf = 'search'
actions_aggs[id_] = (
    search_history
    .filter(pl.col('action_type_id') == id_)
    .filter(pl.col('timestamp').dt.date() <= train_end_date)
    .filter(pl.col('timestamp').dt.date() >= train_end_date - timedelta(days=30 * 4))
    .group_by('user_id')
    .agg(
        pl.count('search_query').cast(pl.Int32).alias(f'num_{suf}'),
        pl.max('timestamp').alias(f'last_{suf}_time'),
        pl.min('timestamp').alias(f'first_{suf}_time'),
    )
    .with_columns([
        (pl.lit(val_start_date) - pl.col(f'last_{suf}_time'))
        .dt.total_days()
        .cast(pl.Int32)
        .alias(f'days_since_last_{suf}'),

        (pl.lit(val_start_date) - pl.col(f'first_{suf}_time'))
        .dt.total_days()
        .cast(pl.Int32)
        .alias(f'days_since_first_{suf}'),
    ])
    .select(
        'user_id',
        f'num_{suf}',
        f'days_since_last_{suf}',
        f'days_since_first_{suf}',
    )
)

In [13]:
actions_aggs.keys()

dict_keys([1, 2, 3, 5, 4])

In [14]:
df = val_target
for _, actions_aggs_df in actions_aggs.items():
    df = (
        df
        .join(actions_aggs_df, on='user_id', how='left')
    )

In [15]:
df

user_id,target,num_products_click,sum_discount_price_click,max_discount_price_click,days_since_last_click,days_since_first_click,num_products_favorite,sum_discount_price_favorite,max_discount_price_favorite,days_since_last_favorite,days_since_first_favorite,num_products_order,sum_discount_price_order,max_discount_price_order,days_since_last_order,days_since_first_order,num_products_to_cart,sum_discount_price_to_cart,max_discount_price_to_cart,days_since_last_to_cart,days_since_first_to_cart,num_search,days_since_last_search,days_since_first_search
i32,i32,i32,f32,f32,i32,i32,i32,f32,f32,i32,i32,i32,f32,f32,i32,i32,i32,f32,f32,i32,i32,i32,i32,i32
10082021,0,2,1483.0,1279.0,81,81,null,null,null,null,null,null,null,null,null,null,1,579.0,579.0,71,71,null,null,null
4248564,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
5712207,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
10492556,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
10932596,1,49,24263.0,3490.0,24,118,null,null,null,null,null,4,996.0,379.0,24,24,146,44190.0,2085.0,24,118,112,13,118
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
10536587,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
72092,1,116,38812.0,1407.0,14,118,1,423.0,423.0,22,22,50,11019.0,1220.0,14,98,290,53881.0,1220.0,14,118,167,14,118
8535398,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [16]:
df_pd = df.to_pandas()

In [17]:
mask = df_pd.user_id % 10 <= 6

In [18]:
df_pd.columns

Index(['user_id', 'target', 'num_products_click', 'sum_discount_price_click',
       'max_discount_price_click', 'days_since_last_click',
       'days_since_first_click', 'num_products_favorite',
       'sum_discount_price_favorite', 'max_discount_price_favorite',
       'days_since_last_favorite', 'days_since_first_favorite',
       'num_products_order', 'sum_discount_price_order',
       'max_discount_price_order', 'days_since_last_order',
       'days_since_first_order', 'num_products_to_cart',
       'sum_discount_price_to_cart', 'max_discount_price_to_cart',
       'days_since_last_to_cart', 'days_since_first_to_cart', 'num_search',
       'days_since_last_search', 'days_since_first_search'],
      dtype='object')

In [19]:
cols = [
    'num_products_click', 
    'sum_discount_price_click', 'max_discount_price_click',
    'days_since_last_click', 'days_since_first_click',
    
    'num_products_favorite',
    'sum_discount_price_favorite', 'max_discount_price_favorite',
    'days_since_last_favorite', 'days_since_first_favorite',
    
    'num_products_order',
    'sum_discount_price_order', 'max_discount_price_order',
    'days_since_last_order', 'days_since_first_order',
    
    'num_products_to_cart',
    'sum_discount_price_to_cart', 'max_discount_price_to_cart',
    'days_since_last_to_cart', 'days_since_first_to_cart',
    
    'num_search',
    'days_since_last_search', 'days_since_first_search'
]

In [20]:
train_pool = catboost.Pool(
    df_pd.loc[mask, cols],
    label=df_pd.loc[mask].target,
)
eval_pool = catboost.Pool(
    df_pd.loc[~mask, cols],
    label=df_pd.loc[~mask].target,
)

In [21]:
train_pool.shape, eval_pool.shape

((1311636, 23), (563320, 23))

In [22]:
params = {
    'iterations': 2000,
    'depth': 7, 
    'learning_rate': 0.1, 
    'random_state': 1,
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
    'task_type': 'GPU',
}

In [23]:
model = catboost.CatBoost(params)
model.fit(
    train_pool, 
    eval_set=eval_pool,
    use_best_model=True,
    verbose=10,
    early_stopping_rounds=50,
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7446812	best: 0.7446812 (0)	total: 64.3ms	remaining: 2m 8s
10:	test: 0.7534201	best: 0.7534201 (10)	total: 164ms	remaining: 29.7s
20:	test: 0.7551414	best: 0.7551414 (20)	total: 256ms	remaining: 24.1s
30:	test: 0.7560869	best: 0.7560869 (30)	total: 341ms	remaining: 21.7s
40:	test: 0.7565900	best: 0.7565900 (40)	total: 435ms	remaining: 20.8s
50:	test: 0.7569643	best: 0.7569643 (50)	total: 535ms	remaining: 20.5s
60:	test: 0.7572643	best: 0.7572643 (60)	total: 630ms	remaining: 20s
70:	test: 0.7574956	best: 0.7574956 (70)	total: 715ms	remaining: 19.4s
80:	test: 0.7576447	best: 0.7576447 (80)	total: 807ms	remaining: 19.1s
90:	test: 0.7577786	best: 0.7577786 (90)	total: 910ms	remaining: 19.1s
100:	test: 0.7579190	best: 0.7579190 (100)	total: 1s	remaining: 18.9s
110:	test: 0.7580191	best: 0.7580338 (108)	total: 1.09s	remaining: 18.6s
120:	test: 0.7581878	best: 0.7581878 (120)	total: 1.18s	remaining: 18.4s
130:	test: 0.7582148	best: 0.7582148 (130)	total: 1.29s	remaining: 18.4s
140:

In [24]:
name = 'baseline1'
model.save_model(f"{name}.bin")

In [25]:
fi = model.get_feature_importance(eval_pool, prettified=True)
fi.head(50)

,Feature Id,Importances
0,sum_discount_price_order,12.274780
1,num_products_order,12.254132
2,max_discount_price_order,11.275465
3,days_since_last_order,10.045279
4,sum_discount_price_to_cart,6.616628
5,days_since_first_order,6.421556
6,num_products_to_cart,5.727590
7,days_since_last_to_cart,4.790685
8,num_products_click,4.779528
9,num_search,4.219211


In [26]:
test_users_submission = (
    pl.read_csv(os.path.join(data_path, 'test_users.csv'))
)

In [27]:
val_end_date, val_end_date - timedelta(days=30 * 4)

(datetime.date(2024, 7, 31), datetime.date(2024, 4, 2))

In [28]:
actions_aggs = {}
actions_id_to_suf = {
    1: "click",
    2: "favorite",
    3: "order",
    5: "to_cart",
}
for id_, suf in actions_id_to_suf.items():
    aggs = (
        actions_history
        .filter(pl.col('timestamp').dt.date() <= val_end_date)
        .filter(pl.col('timestamp').dt.date() >= val_end_date - timedelta(days=30 * 4))
        .filter(pl.col('action_type_id') == id_)
        .join(
            product_information
            .select('product_id', 'discount_price'),
            on='product_id',
        )
        .group_by('user_id')
        .agg(
            pl.count('product_id').cast(pl.Int32).alias(f'num_products_{suf}'),
            pl.sum('discount_price').cast(pl.Float32).alias(f'sum_discount_price_{suf}'),
            pl.max('discount_price').cast(pl.Float32).alias(f'max_discount_price_{suf}'),
            pl.max('timestamp').alias(f'last_{suf}_time'),
            pl.min('timestamp').alias(f'first_{suf}_time'),
        )
        .with_columns([
            (pl.lit(test_start_date) - pl.col(f'last_{suf}_time'))
            .dt.total_days()
            .cast(pl.Int32)
            .alias(f'days_since_last_{suf}'),
            
            (pl.lit(test_start_date) - pl.col(f'first_{suf}_time'))
            .dt.total_days()
            .cast(pl.Int32)
            .alias(f'days_since_first_{suf}'),
        ])
        .select(
            'user_id',
            f'num_products_{suf}',
            f'sum_discount_price_{suf}',
            f'max_discount_price_{suf}',
            f'days_since_last_{suf}',
            f'days_since_first_{suf}',
        )
    )
    actions_aggs[id_] = aggs

In [29]:
# search_aggs
id_ = 4
suf = 'search'
actions_aggs[id_] = (
    search_history
    .filter(pl.col('action_type_id') == id_)
    .filter(pl.col('timestamp').dt.date() <= val_end_date)
    .filter(pl.col('timestamp').dt.date() >= val_end_date - timedelta(days=30 * 4))
    .group_by('user_id')
    .agg(
        pl.count('search_query').cast(pl.Int32).alias(f'num_{suf}'),
        pl.max('timestamp').alias(f'last_{suf}_time'),
        pl.min('timestamp').alias(f'first_{suf}_time'),
    )
    .with_columns([
        (pl.lit(test_start_date) - pl.col(f'last_{suf}_time'))
        .dt.total_days()
        .cast(pl.Int32)
        .alias(f'days_since_last_{suf}'),

        (pl.lit(test_start_date) - pl.col(f'first_{suf}_time'))
        .dt.total_days()
        .cast(pl.Int32)
        .alias(f'days_since_first_{suf}'),
    ])
    .select(
        'user_id',
        f'num_{suf}',
        f'days_since_last_{suf}',
        f'days_since_first_{suf}',
    )
)

In [30]:
df = test_users_submission
for _, actions_aggs_df in actions_aggs.items():
    df = (
        df
        .join(actions_aggs_df, on='user_id', how='left')
    )

In [31]:
df_pd = df.to_pandas()

In [32]:
df_pd.shape

(2068424, 24)

In [33]:
df_pd['predict'] = model.predict(df_pd[cols], prediction_type="Probability")[:, 1]

In [34]:
df_pd[['user_id', 'predict']]

,user_id,predict
0,1342,0.196968
1,9852,0.814203
2,10206,0.222463
3,11317,0.220989
4,13289,0.633537
...,...,...
2068419,11157283,0.224383
2068420,11160395,0.133542
2068421,11165052,0.590410
2068422,11168218,0.529186


In [35]:
df_pd[['user_id', 'predict']].to_csv('baseline1_submission.csv', index=False)